In [ ]:
import os, json, random
import pandas as pd
from openai import OpenAI
from sklearn.metrics import classification_report, confusion_matrix

from ddi.data import build_human, ALL_LABELS          # ["NONE","ADVISE","EFFECT","INT","MECHANISM"]
from ddi.manifest import write_dataset, load_dataset
from ddi.vocab import build_vocab
from ddi.prompt import (make_specs, make_sample_fn, make_verifier,
                        prompt_fingerprint)
from ddi.synth import generate_raw, build_dataset_from_raw, RAW
from ddi.run import run_training

In [ ]:
# same client used for generation and verification, but with different model/temperature settings (see GEN_CFG, VER_CFG below)

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=5, timeout=120.0)
vocab = build_vocab()

In [ ]:
# run a simple ping to check the connection to the LLM API
client.responses.parse(model="gpt-oss-120b",
                        input=[{"role":"user","content":"ping"}])

In [ ]:
# train, dev, val = build_human()
# dev_id = write_dataset(dev, provenance="human:dev", seed=42, notes="dev - iterate here")
# val_id = write_dataset(val, provenance="human:val", seed=42, notes="val - finalists only")
# print(dev_id, val_id)

dev_id = "20260723-032552-e46d2b"
val_id = "20260723-032553-e79bce"

In [ ]:
SYNTH_VERSION = "v8"
GEN_CFG  = {"model": "gpt-oss-120b", "temperature": 0.7,
            "reasoning_effort": "low", "max_output_tokens": 3000}
SPEC_CFG = {"n": 50, "seed": 0}
VER_CFG  = {"model": "gpt-oss-120b", "temperature": 0,
            "reasoning_effort": "high", "max_output_tokens": 3000} 
TRAIN_CFG = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
             "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
             "neg_ratio": None, "seed": 0, "dataset": "synthetic"}

In [ ]:
# the verdict reader (new glue -> move to ddi/verify.py once stable)

# Reads a verifier generate_raw run. `spec` is the instance dict that went in
# ({text,label,source,sent_id}); `sample` is the Verdict. For human dev, `label`
# is gold; for synthetic, `label` is the intended (requested) class.
def load_verdicts(gen_id):
    rows = []
    for line in (RAW / f"{gen_id}.jsonl").read_text().splitlines():
        if not line:
            continue
        rec = json.loads(line)
        inst, smp = rec["spec"], rec["sample"]
        rows.append({"text": inst["text"], "ref": inst["label"],
                     "source": inst.get("source"),
                     "pred": (smp or {}).get("label"),
                     "why":  (smp or {}).get("justification"),
                     "error": rec["error"]})
    return pd.DataFrame(rows)

In [ ]:
# generate synthetic data (specs) 

specs     = make_specs(vocab=vocab, **SPEC_CFG)
sample_fn = make_sample_fn(client, **GEN_CFG)
generate_raw(specs, sample_fn, gen_id=SYNTH_VERSION, max_workers=16)

In [ ]:
# build synthetic dataset from raw JSONL, with stats

synth_id, stats = build_dataset_from_raw(
    SYNTH_VERSION,
    generator={**GEN_CFG, "specs": SPEC_CFG, "prompt_sha": prompt_fingerprint()},
    vocab_source=vocab.fingerprint(), seed=SPEC_CFG["seed"])
print(stats)

In [ ]:
inst, man = load_dataset(synth_id)
for r in inst[:10]:
    print(f"{r['label']:10s} {r['source']:10s} {r['text']}")
dist = man["label_distribution"]
print(f"NONE ratio: {dist['NONE']/sum(dist.values()):.3f}  ({dist})")

In [ ]:
dev_inst, _ = load_dataset(dev_id)
verify_fn = make_verifier(client, **VER_CFG)
generate_raw(dev_inst, verify_fn, gen_id="verify-humandev-v3-high", max_workers=16)

In [ ]:
# is the verifier a competent annotator? (vs gold)

d = load_verdicts("verify-humandev-v3-high")
print(f"{d.error.notna().sum()} errored of {len(d)}")
d = d[d.error.isna()]
print(classification_report(d.ref, d.pred, labels=ALL_LABELS, zero_division=0))
# rows = gold, cols = verifier. Watch the MECHANISM->EFFECT cell: your EFFECT def
# includes PD mechanisms, so that boundary is where drift hides.
print(pd.DataFrame(confusion_matrix(d.ref, d.pred, labels=ALL_LABELS),
                   index=ALL_LABELS, columns=ALL_LABELS))

In [ ]:
# run the verifier on the synthetic dataset

# NOTE: ~C(n_drugs,2) instances per spec, so 2000 specs ~= 12k instances (~30 min).
# Verifying the NONEs is the point — that's the smuggled-interaction check.
generate_raw(inst, verify_fn, gen_id=f"verify-{synth_id}", max_workers=16)

In [ ]:
# fidelity — but read it through CELL 10's verifier error
s = load_verdicts(f"verify-{synth_id}")
s = s[s.error.isna()]
pos = s[s.ref != "NONE"]
neg = s[s.ref == "NONE"]
# (a) do positives hold up? of sentences intended class C, fraction the blind verifier confirms:
for c in ["ADVISE", "EFFECT", "INT", "MECHANISM"]:
    sub = pos[pos.ref == c]
    print(f"{c:10s} fidelity {sub.pred.eq(c).mean():.3f}  (n={len(sub)})")
# (b) are the free negatives clean? fraction of asserted-NONE that read as an interaction:
print(f"\nsmuggled interactions in NONE: {neg.pred.ne('NONE').mean():.3f}  (n={len(neg)})")
print(pd.DataFrame(confusion_matrix(s.ref, s.pred, labels=ALL_LABELS),
                   index=ALL_LABELS, columns=ALL_LABELS))

In [ ]:
run_id, metrics = run_training(synth_id, dev_id, TRAIN_CFG, notes=f"synthetic-only {SYNTH_VERSION}")
print(metrics["micro_f1_pos"])

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal

class Ping(BaseModel):
    label: Literal["ok"]
    reason: str

client = OpenAI(base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
                api_key=os.environ["IDA_LLM_API_KEY"], max_retries=1, timeout=60.0)

MODEL = "ministral-3-8b-reasoning-2512"      # swap to test a stand-in

resp = client.responses.parse(
    model=MODEL,
    input=[{"role":"system","content":"Return only a JSON object with keys label and reason."},
           {"role":"user","content":"ping"}],
    temperature=0,
    max_output_tokens=2000,
    reasoning={"effort": "high"},
)
print(resp.output_text)

In [ ]:
raw = "label{\t\t\"label\": \"ok\", \"reason\": \"pong\"\n}"
import json
json.loads(raw[raw.index("{"):])